In [ ]:
# ============================================================
# Cell 1: Production Dependencies
# ============================================================
!pip -q install torch torchvision open-clip-torch insightface onnxruntime albumentations opencv-python-headless Pillow numpy


In [ ]:
# ============================================================
# Cell 2: Core Model Architecture (model.py)
# ============================================================
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import open_clip

@dataclass
class ModelConfig:
    hidden1: int = 512
    hidden2: int = 256
    feature_dropout: float = 0.1
    fc_dropout: float = 0.4

class FrozenFaceBackbone(nn.Module):
    embedding_dim = 512
    def __init__(self):
        super().__init__()
        self.model = models.resnet50(weights=None)
        self.model.fc = nn.Linear(self.model.fc.in_features, self.embedding_dim)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

        self.register_buffer("mean", torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.5, 0.5, 0.5]).view(1, 3, 1, 1))

    def forward(self, x01):
        x = (x01 - self.mean) / self.std
        emb = self.model(x)
        return F.normalize(emb, dim=1)


class FrozenCLIPBackbone(nn.Module):
    embedding_dim = 768
    def __init__(self):
        super().__init__()
        model, _, _ = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
        self.model = model.visual
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

        clip_mean = [0.48145466, 0.4578275, 0.40821073]
        clip_std = [0.26862954, 0.26130258, 0.27577711]
        self.register_buffer("mean", torch.tensor(clip_mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(clip_std).view(1, 3, 1, 1))

    def forward(self, x01):
        x = (x01 - self.mean) / self.std
        feat = self.model(x)
        return F.normalize(feat.float(), dim=1)


class FusionHead(nn.Module):
    def __init__(self, face_dim: int, clip_dim: int, hidden1: int, hidden2: int, feat_drop: float, fc_drop: float):
        super().__init__()
        self.input_norm = nn.LayerNorm(face_dim + clip_dim)
        self.feature_dropout = nn.Dropout(feat_drop)
        self.net = nn.Sequential(
            nn.Linear(face_dim + clip_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.GELU(),
            nn.Dropout(fc_drop),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.GELU(),
            nn.Dropout(fc_drop),
            nn.Linear(hidden2, 1),
        )

    def forward(self, face_emb, clip_emb):
        x = torch.cat([face_emb, clip_emb], dim=1)
        return self.net(self.feature_dropout(self.input_norm(x))).squeeze(1)


class FusionModel(nn.Module):
    def __init__(self, cfg: ModelConfig = ModelConfig()):
        super().__init__()
        self.face_backbone = FrozenFaceBackbone()
        self.clip_backbone = FrozenCLIPBackbone()
        self.head = FusionHead(
            self.face_backbone.embedding_dim,
            self.clip_backbone.embedding_dim,
            cfg.hidden1, cfg.hidden2, cfg.feature_dropout, cfg.fc_dropout
        )

    def forward(self, x_384):
        clip_input = F.interpolate(x_384, size=(224, 224), mode='bicubic', antialias=True)
        crop_size = int(x_384.shape[-1] * 0.8)
        start = (x_384.shape[-1] - crop_size) // 2
        face_crop = x_384[:, :, start:start + crop_size, start:start + crop_size]
        face_input = F.interpolate(face_crop, size=(112, 112), mode='bicubic', antialias=True)
        return self.head(self.face_backbone(face_input), self.clip_backbone(clip_input))


In [ ]:
# ============================================================
# Cell 3: Preprocessing & Cropping Pipeline (preprocessing.py)
# ============================================================
import numpy as np
import cv2
from PIL import Image
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
from insightface.app import FaceAnalysis

class ProductionFacePreprocessor:
    def __init__(self, device: str = "cpu", offline_margin: float = 1.4, master_size: int = 384):
        self.device = device
        self.offline_margin = offline_margin
        self.master_size = master_size
        self.fallback_center_crop_frac = 0.8

        # Initialize RetinaFace ONCE on startup
        self.detector = FaceAnalysis(
            name="buffalo_l",
            allowed_modules=["detection"],
            providers=["CUDAExecutionProvider"] if device == "cuda" else ["CPUExecutionProvider"],
        )
        self.detector.prepare(ctx_id=0 if device == "cuda" else -1, det_size=(640, 640))

        # Albumentations tensor transform
        self.transform = A.Compose([ToTensorV2()])

    def _expand_and_square_box(self, box, margin):
        x1, y1, x2, y2 = box
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        side = max(x2 - x1, y2 - y1) * margin
        half = side / 2
        return cx - half, cy - half, cx + half, cy + half

    def _crop_with_reflect_pad(self, image_pil, box):
        img_w, img_h = image_pil.size
        x1, y1, x2, y2 = [int(round(v)) for v in box]
        pad_left = max(0, -x1)
        pad_top = max(0, -y1)
        pad_right = max(0, x2 - img_w)
        pad_bottom = max(0, y2 - img_h)

        if any([pad_left, pad_top, pad_right, pad_bottom]):
            arr = np.array(image_pil)
            arr = np.pad(arr, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), mode="reflect")
            image_pil = Image.fromarray(arr)
            x1, y1, x2, y2 = x1 + pad_left, y1 + pad_top, x2 + pad_left, y2 + pad_top

        return image_pil.crop((x1, y1, x2, y2))

    def process_pil_image(self, image_pil: Image.Image) -> torch.Tensor:
        """Takes a PIL Image and returns a normalized (1, 3, 384, 384) tensor ready for FusionModel."""
        image_pil = image_pil.convert("RGB")
        bgr = np.array(image_pil)[:, :, ::-1]
        faces = self.detector.get(bgr)

        if not faces:
            img_w, img_h = image_pil.size
            side = min(img_w, img_h) * self.fallback_center_crop_frac
            cx, cy = img_w / 2, img_h / 2
            inner_box = (cx - side / 2, cy - side / 2, cx + side / 2, cy + side / 2)
        else:
            best = max(faces, key=lambda f: f.det_score)
            x1, y1, x2, y2 = best.bbox
            inner_box = (float(x1), float(y1), float(x2), float(y2))

        box = self._expand_and_square_box(inner_box, self.offline_margin)
        cropped = self._crop_with_reflect_pad(image_pil, box)
        resized_crop = cropped.resize((self.master_size, self.master_size), Image.Resampling.LANCZOS)

        img_array = np.array(resized_crop)
        tensor = self.transform(image=img_array)["image"].float() / 255.0
        return tensor.unsqueeze(0) # Add batch dimension (1, 3, 384, 384)


In [ ]:
# ============================================================
# Cell 4: Production Inference Handler (inference_engine.py)
# ============================================================
import io
import torch
from PIL import Image
from typing import Dict, Any, Union

class AIFaceDetectionService:
    def __init__(self, checkpoint_path: str, optimal_temperature: float = 0.7712, device: str = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.optimal_temp = optimal_temperature

        # 1. Load Preprocessor
        print(f"[Init] Setting up Preprocessor on {self.device}...")
        self.preprocessor = ProductionFacePreprocessor(device=self.device)

        # 2. Load Model & Weights
        print(f"[Init] Loading Model Weights from {checkpoint_path}...")
        self.model = FusionModel(ModelConfig()).to(self.device)
        checkpoint = torch.load(checkpoint_path, map_location=self.device)

        if "model_state_dict" in checkpoint:
            self.model.load_state_dict(checkpoint["model_state_dict"])
        else:
            self.model.load_state_dict(checkpoint)

        self.model.eval()
        print("[Init] AI Detection Service successfully initialized and ready!")

    @torch.no_grad()
    def predict(self, image_input: Union[str, bytes, Image.Image]) -> Dict[str, Any]:
        """
        Accepts a file path, raw image bytes, or PIL Image.
        Returns a structured JSON payload for frontend consumption.
        """
        # Parse image source
        if isinstance(image_input, str):
            image_pil = Image.open(image_input)
        elif isinstance(image_input, bytes):
            image_pil = Image.open(io.BytesIO(image_input))
        elif isinstance(image_input, Image.Image):
            image_pil = image_input
        else:
            raise ValueError("Unsupported image input type. Pass a file path, bytes, or PIL Image.")

        # 1. Preprocess Image
        tensor = self.preprocessor.process_pil_image(image_pil).to(self.device)

        # 2. TTA Forward Pass (Original + Horizontal Flip)
        logits_normal = self.model(tensor)
        logits_flipped = self.model(torch.flip(tensor, dims=[-1]))
        logits = (logits_normal + logits_flipped) / 2.0

        # 3. Temperature Scaling & Probability
        calibrated_logits = logits / self.optimal_temp
        prob = torch.sigmoid(calibrated_logits).item()

        # 4. Format Structured API Response
        is_ai = prob >= 0.5
        confidence = prob if is_ai else (1.0 - prob)

        return {
            "status": "success",
            "is_ai": is_ai,
            "label": "AI-Generated" if is_ai else "Real Photo",
            "confidence": round(confidence * 100, 2),
            "raw_ai_probability": round(prob, 4)
        }

# ============================================================
# Example Usage for Backend Web Server (FastAPI / Flask)
# ============================================================

# Step A: Instantiate Service Once on App Startup
CHECKPOINT_PATH = "/content/drive/MyDrive/AI_Image_Detection/checkpoints_final/best_final_model.pt"
service = AIFaceDetectionService(checkpoint_path=CHECKPOINT_PATH, optimal_temperature=0.7712)

# Step B: Call 'service.predict()' inside API endpoints
# response = service.predict("path/to/test_image.jpg")
# print(response)
# Output: {'status': 'success', 'is_ai': False, 'label': 'Real Photo', 'confidence': 99.85, 'raw_ai_probability': 0.0015}
